In [11]:
!pip install langchain langchain-community sentence-transformers faiss-cpu

In [13]:
from dotenv import load_dotenv
import os
import pandas as pd
from sqlalchemy import create_engine

from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [14]:
import pandas as pd

feedback = pd.read_csv("blinkit_customer_feedback.csv")
customers = pd.read_csv("blinkit_customers.csv")

In [15]:
customer_feedback_view = feedback.merge(
    customers[["customer_id", "area"]],
    on="customer_id",
    how="left"
)

In [16]:
customer_feedback_view = customer_feedback_view[
    ["feedback_id", "feedback_text", "feedback_category", "sentiment", "area"]
]

In [17]:
print(len(customer_feedback_view))
customer_feedback_view.head()

5000


,feedback_id,feedback_text,feedback_category,sentiment,area
0,2234710,"It was okay, nothing special.",Delivery,Neutral,Allahabad
1,5450964,The order was incorrect.,App Experience,Negative,Thrissur
2,482108,"It was okay, nothing special.",App Experience,Neutral,Vellore
3,4823104,The product met my expectations.,App Experience,Neutral,Gaya
4,3537464,Product was damaged during delivery.,Delivery,Negative,Asansol


In [19]:
from langchain_core.documents import Document

documents = [
    Document(page_content=row["feedback_text"])
    for _, row in customer_feedback_view.iterrows()
]

print("Documents created:", len(documents))

Documents created: 5000


In [20]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(documents, embeddings)

print("Vector DB ready")

/tmp/ipykernel_378/3249945339.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB ready


In [21]:
results = vectorstore.similarity_search("late delivery")

print(results[0].page_content)

Delivery was late and I was unhappy.


In [23]:
sentiment_summary = customer_feedback_view["sentiment"].value_counts()

print(sentiment_summary)

sentiment
Neutral     1738
Negative    1642
Positive    1620
Name: count, dtype: int64
